In [79]:
import torch.nn as nn
import torch

In [80]:
class LoRALinear(nn.Module):
  def __init__(self, in_features, out_features, r=50, alpha=1.0, bias=True):
    super().__init__()
    self.weight = nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
    self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=False) if bias is not None else None
    self.A = nn.Parameter(torch.randn(in_features, r))
    self.B = nn.Parameter(torch.zeros(r, out_features))

    self.scaling = alpha/r
  def forward(self, x):
    # x.shape = (samples, in_features)
    # (samples, in) @ (in, out)
    base = x @ self.weight.T
    lora = (x @ self.A) @ self.B
    out = base + self.scaling * lora
    if self.bias is not None:
      out = out + self.bias
    return out

In [81]:
x = torch.rand(100, 60) # samples = 100, in_features = 60
linear_lora = LoRALinear(in_features=x.shape[1], out_features=20, bias=False)
output = linear_lora.forward(x) # shape = (100, 20)

In [82]:
output.shape

torch.Size([100, 20])

In [83]:
for name, param in linear_lora.named_parameters():
  print(f'name = {name}, param.grad = {param.requires_grad}')

name = weight, param.grad = False
name = bias, param.grad = False
name = A, param.grad = True
name = B, param.grad = True


In [84]:
class Model(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.fc1 = nn.Linear(config['in_features'], config['hidden_size1'])
    self.fc2 = nn.Linear(config['hidden_size1'], config['hidden_size2'])
    self.fc3 = nn.Linear(config['hidden_size2'], config['out_features'])

    self.tanh = nn.Tanh()

  def forward(self, x):
    x = self.tanh(self.fc1(x))
    x = self.tanh(self.fc2(x))
    x = self.fc3(x)
    return x

In [85]:
model = Model({'in_features': 100, 'hidden_size1': 50, 'hidden_size2': 70, 'out_features': 20})

In [86]:
for name, modules in model.named_modules():
  print(f'{name}:{modules}')

:Model(
  (fc1): Linear(in_features=100, out_features=50, bias=True)
  (fc2): Linear(in_features=50, out_features=70, bias=True)
  (fc3): Linear(in_features=70, out_features=20, bias=True)
  (tanh): Tanh()
)
fc1:Linear(in_features=100, out_features=50, bias=True)
fc2:Linear(in_features=50, out_features=70, bias=True)
fc3:Linear(in_features=70, out_features=20, bias=True)
tanh:Tanh()


In [87]:
cash_weight = model.fc1.weight
cash_bias = model.fc1.bias

In [88]:
cash_weight

Parameter containing:
tensor([[-0.0620,  0.0035,  0.0414,  ...,  0.0027, -0.0906, -0.0869],
        [-0.0137, -0.0920, -0.0926,  ..., -0.0648, -0.0089, -0.0926],
        [-0.0674,  0.0070, -0.0077,  ...,  0.0874,  0.0130,  0.0318],
        ...,
        [-0.0308, -0.0029,  0.0912,  ..., -0.0852,  0.0595,  0.0566],
        [-0.0534, -0.0032, -0.0498,  ..., -0.0457,  0.0379,  0.0966],
        [-0.0655, -0.0511,  0.0096,  ..., -0.0398,  0.0132,  0.0879]],
       requires_grad=True)

In [89]:
cash_bias

Parameter containing:
tensor([-0.0721, -0.0761,  0.0667,  0.0738,  0.0043, -0.0909,  0.0948,  0.0559,
        -0.0035,  0.0718, -0.0485,  0.0095, -0.0598,  0.0075,  0.0219, -0.0083,
        -0.0560,  0.0478, -0.0215,  0.0370, -0.0672, -0.0884, -0.0012, -0.0430,
         0.0012,  0.0070,  0.0889, -0.0730, -0.0462, -0.0469, -0.0607,  0.0697,
         0.0335, -0.0098,  0.0639,  0.0109,  0.0069,  0.0521, -0.0973,  0.0821,
         0.0780, -0.0749, -0.0276, -0.0813,  0.0799, -0.0794,  0.0929, -0.0663,
         0.0271,  0.0482], requires_grad=True)

In [90]:
model.fc1 = linear_lora

In [91]:
for name, modules in model.named_modules():
  print(f'{name}:{modules}')

:Model(
  (fc1): LoRALinear()
  (fc2): Linear(in_features=50, out_features=70, bias=True)
  (fc3): Linear(in_features=70, out_features=20, bias=True)
  (tanh): Tanh()
)
fc1:LoRALinear()
fc2:Linear(in_features=50, out_features=70, bias=True)
fc3:Linear(in_features=70, out_features=20, bias=True)
tanh:Tanh()


In [92]:
model.fc1.weight.data = cash_weight.data
model.fc1.weight.requires_grad = False
model.fc1.bias.data = cash_bias.data
model.fc1.bias.requires_grad = False

In [93]:
for name, param in model.named_parameters():
  print(f'{name}:{param.requires_grad}')

fc1.weight:False
fc1.bias:False
fc1.A:True
fc1.B:True
fc2.weight:True
fc2.bias:True
fc3.weight:True
fc3.bias:True


In [94]:
setattr(model, 'fc2', linear_lora)

In [95]:
model.fc2

LoRALinear()

In [96]:
for name, module in model.named_modules():
  if isinstance(module, nn.Linear):
    lora = LoRALinear(
        module.in_features,
        module.out_features,
        bias=module.bias
    )
    lora.weight.data = module.weight.data
    lora.bias.data = module.bias.data
    setattr(model, name, lora)

In [97]:
model

Model(
  (fc1): LoRALinear()
  (fc2): LoRALinear()
  (fc3): LoRALinear()
  (tanh): Tanh()
)

In [98]:
for name, param in model.named_parameters():
  print(f'{name}:{param.requires_grad}')

fc1.weight:False
fc1.bias:False
fc1.A:True
fc1.B:True
fc3.weight:False
fc3.bias:False
fc3.A:True
fc3.B:True
